# MicroGrad 计算图追踪（默会知识自学版）

本 Notebook 的目标不是只会调用 `draw_dot`，而是建立你对“计算图 + 反向传播流向”的直觉。

按 Michael Polanyi 默会知识理论学习：
- 显性知识：你能解释每个函数、每条边、每个梯度。
- 默会知识：你能在运行前大致预判图结构和梯度方向。

建议每格都执行这 4 步：
1. 先画草图：先手画你预测的计算图。
2. 再运行：对比程序图和你的草图。
3. 做微改：一次只改一个操作（如 relu 改成加法）。
4. 写复盘：记录“规则 + 直觉”各一句。

In [ ]:
# ===== 1) 图可视化依赖 =====
# 系统层面需要 graphviz（命令行工具），Python 层面需要 graphviz 包
# macOS: brew install graphviz
# Python: pip install graphviz
from graphviz import Digraph

# 学习提示：
# - 如果这里只安装了 Python 包但没安装系统 graphviz，render 可能失败。
# - 默会知识目标：你会形成“可视化报错先查环境而不是先怀疑算法”的工程直觉。

In [ ]:
# ===== 2) 导入带自动求导能力的标量 Value =====
from micrograd.engine import Value

# Value 内部通常包含：
# - data: 前向值
# - grad: 反向梯度
# - _prev: 其前驱节点（用于追踪计算图）
# - _op: 当前节点由什么操作得到（如 +、*、relu）

In [ ]:
# ===== 3) 追踪并绘制计算图 =====
def trace(root):
    """
    从输出节点 root 反向遍历，收集：
    - nodes: 所有参与该输出计算的节点
    - edges: 子节点 -> 父节点 的有向边
    """
    nodes, edges = set(), set()

    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)

    build(root)
    return nodes, edges


def draw_dot(root, format='svg', rankdir='LR'):
    """
    format: png | svg | ...
    rankdir: TB（上到下）| LR（左到右）
    """
    assert rankdir in ['LR', 'TB']
    nodes, edges = trace(root)
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir})

    # 每个 Value 节点显示 data 和 grad
    # 如果该节点由某个操作产生，则额外画一个操作节点与其连接
    for n in nodes:
        dot.node(
            name=str(id(n)),
            label="{ data %.4f | grad %.4f }" % (n.data, n.grad),
            shape='record',
        )
        if n._op:
            dot.node(name=str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))

    # child -> op(node) 表示“子节点参与了父节点的运算”
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

# 自学任务：
# 1) 把 rankdir 改成 'TB'，比较你的阅读速度。
# 2) 先不运行，手动画一个 x*2+1 的图，再与 draw_dot 对照。
# 3) 在 label 里加入 _op 信息，体会图可读性如何变化。

In [ ]:
# ===== 4) 示例一：一维简单表达式 =====
# 表达式：y = relu(2x + 1)
x = Value(1.0)
y = (x * 2 + 1).relu()

# 反向传播：把 y 对图中所有节点的梯度算出来
y.backward()

# 画图查看 data/grad 与操作连接关系
draw_dot(y)

# 先预测再运行：
# - 当 x=1.0 时，2x+1=3，relu 激活，梯度应能向前传递。
# - 试试 x=-1.0，观察 relu 截断后梯度是否变为 0。

In [ ]:
# ===== 5) 示例二：二维神经元 =====
import random
from micrograd import nn

# 固定随机种子，保证神经元初始参数可复现
random.seed(1337)

# 创建一个 2 输入神经元
n = nn.Neuron(2)
x = [Value(1.0), Value(-2.0)]

# 前向 + 反向
y = n(x)
y.backward()

# 可视化该神经元对应的计算图
dot = draw_dot(y)
dot

# 自学任务：
# 1) 把输入改成 [2.0, -1.0]，预测输出节点 data 的变化方向。
# 2) 连续改 3 组输入，观察哪些节点的 grad 最敏感。
# 3) 尝试口述：权重、偏置分别在图中如何影响输出。

In [ ]:
# ===== 6) 导出图文件 =====
# 渲染并保存为本地文件（默认会生成 gout 和对应格式文件）
dot.render('gout')

# 如果你想指定格式：
# draw_dot(y, format='png').render('gout_png')
# draw_dot(y, format='svg').render('gout_svg')

# 学习提示：把不同输入下的图都导出，对比 grad 数值变化，
# 你会逐步获得“看图就能猜梯度传播是否正常”的默会能力。

## 7) 复盘模板（建议每次改动后填写）

1. 我这次只改了哪个变量或操作？
2. 运行前我的预测是什么？
3. 实际图结构与梯度流向有什么差异？
4. 显性知识：我能明确写出的规则是什么？
5. 默会知识：我新形成了什么判断手感？
6. 下一步最小实验是什么？

长期目标：从“看得懂图”进阶到“运行前能预判图”。